In [3]:
import cv2
import numpy as np
import math
import matplotlib.pyplot as plt
from pathlib import Path

# =========================
# GIVEN PARAMETERS
# =========================
THETA_DEG = 67.5           # guide angle
R_MM = 45.0                # disc radius
L_MM = 250.0               # rod length AB
REF_DIST_MM = 79.0         # red–blue fixed distance
VIDEO_PATH = "output.mp4"  # your video file

# =========================
# 0. READ VIDEO
# =========================
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
dt = 1.0 / fps

frames = []
while True:
    ok, fr = cap.read()
    if not ok:
        break
    # convert to RGB for uniform processing
    frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
cap.release()

n_frames = len(frames)
print("frames:", n_frames, "fps:", fps)

# =========================
# helpers
# =========================
def biggest_centroid(mask):
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    M = cv2.moments(c)
    if M["m00"] == 0:
        return None
    return np.array([M["m10"]/M["m00"], M["m01"]/M["m00"]], float)

# HSV ranges (these worked on your frames I looked at)
HSV_RANGES = {
    "red1": ((0, 80, 80), (10, 255, 255)),
    "red2": ((170, 80, 80), (180, 255, 255)),
    "blue": ((90, 80, 80), (130, 255, 255)),
}

# =========================
# 1. SCALE FROM FRAME 0
# =========================
frame0 = frames[0]
hsv0 = cv2.cvtColor(frame0, cv2.COLOR_RGB2HSV)

mask_red1 = cv2.inRange(hsv0, np.array(HSV_RANGES["red1"][0]), np.array(HSV_RANGES["red1"][1]))
mask_red2 = cv2.inRange(hsv0, np.array(HSV_RANGES["red2"][0]), np.array(HSV_RANGES["red2"][1]))
mask_red = cv2.bitwise_or(mask_red1, mask_red2)
mask_blue = cv2.inRange(hsv0, np.array(HSV_RANGES["blue"][0]), np.array(HSV_RANGES["blue"][1]))

red_px = biggest_centroid(mask_red)
blue_px = biggest_centroid(mask_blue)
if red_px is None or blue_px is None:
    raise RuntimeError("Could not find the fixed red/blue markers in frame 0.")

dist_px = np.linalg.norm(red_px - blue_px)
scale_mm_per_px = REF_DIST_MM / dist_px
print("scale (mm/px):", scale_mm_per_px)

# =========================
# 2. DISC CENTRE ON SPARSE FRAMES
# =========================
# detect disc once in frame 0 to get a good guess
gray0 = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)
gray0 = cv2.medianBlur(gray0, 5)
expected_radius_px = R_MM / scale_mm_per_px

circs = cv2.HoughCircles(
    gray0, cv2.HOUGH_GRADIENT, dp=1.2, minDist=200,
    param1=100, param2=30,
    minRadius=int(expected_radius_px * 0.7),
    maxRadius=int(expected_radius_px * 1.4),
)
if circs is None:
    raise RuntimeError("Could not detect disc in frame 0.")

circs = np.squeeze(circs)
if circs.ndim == 1:
    disc0 = np.array([circs[0], circs[1]], float)
else:
    disc0 = min(circs, key=lambda c: abs(c[2] - expected_radius_px))[:2]
disc_centres = {0: disc0}

# detect every 8th frame only, interpolate others
key_frames = list(range(0, n_frames, 8))
if (n_frames - 1) not in key_frames:
    key_frames.append(n_frames - 1)

last_center = disc0.copy()
for k in key_frames[1:]:
    gray = cv2.cvtColor(frames[k], cv2.COLOR_RGB2GRAY)
    gray = cv2.medianBlur(gray, 5)
    circs = cv2.HoughCircles(
        gray, cv2.HOUGH_GRADIENT, dp=1.2, minDist=200,
        param1=100, param2=30,
        minRadius=int(expected_radius_px * 0.7),
        maxRadius=int(expected_radius_px * 1.4),
    )
    if circs is not None:
        circs = np.squeeze(circs)
        if circs.ndim == 1:
            c_best = np.array([circs[0], circs[1]], float)
        else:
            c_best = min(
                circs,
                key=lambda c: (c[0] - last_center[0]) ** 2 + (c[1] - last_center[1]) ** 2,
            )[:2]
        last_center = c_best
    disc_centres[k] = last_center.copy()

# interpolate to every frame
O_px_all = []
for i in range(n_frames):
    if i in disc_centres:
        O_px_all.append(disc_centres[i])
    else:
        # find two surrounding keyframes
        before = max([kf for kf in key_frames if kf <= i])
        after = min([kf for kf in key_frames if kf >= i])
        c1 = disc_centres[before]
        c2 = disc_centres[after]
        w = (i - before) / (after - before) if after != before else 0.0
        O_px_all.append((1 - w) * c1 + w * c2)
O_px_all = np.array(O_px_all)

# =========================
# 3. MAIN LOOP: A(t), B(t), γ, α, s
# =========================
theta_rad = math.radians(THETA_DEG)
guide_angle = math.pi - theta_rad   # because slider is to the left
g = np.array([math.cos(guide_angle), math.sin(guide_angle)], float)

times = []
gamma_exp = []
gamma_th = []
s_exp = []
s_th = []
alpha_exp = []
alpha_th = []
xO_mm_list = []
A_mm_list = []

disc_x0_mm = None

for i, frame in enumerate(frames):
    hsv = cv2.cvtColor(frame, cv2.COLOR_RGB2HSV)
    mask_blue = cv2.inRange(hsv, np.array(HSV_RANGES["blue"][0]), np.array(HSV_RANGES["blue"][1]))
    A_px = biggest_centroid(mask_blue)
    if A_px is None:
        # skip if we miss blue in this frame
        continue

    O_px = O_px_all[i]
    # A and O in mm with O as origin, y up
    A_mm = np.array(
        [
            (A_px[0] - O_px[0]) * scale_mm_per_px,
            (O_px[1] - A_px[1]) * scale_mm_per_px,
        ],
        float,
    )
    xO_mm = O_px[0] * scale_mm_per_px

    if disc_x0_mm is None:
        disc_x0_mm = xO_mm

    # --- reconstruct B by circle-circle intersection ---
    d_vec = -A_mm           # vector from A to O (because O is at (0,0))
    dist_AO = np.linalg.norm(d_vec)
    a = (L_MM**2 - R_MM**2 + dist_AO**2) / (2 * dist_AO)
    h_sq = L_MM**2 - a**2
    h_val = math.sqrt(h_sq) if h_sq > 0 else 0.0
    P2 = A_mm + a * d_vec / dist_AO
    perp = np.array([-d_vec[1], d_vec[0]], float) / dist_AO
    B1 = P2 + h_val * perp
    B2 = P2 - h_val * perp
    # choose the intersection above the disc (larger y)
    B_mm = B1 if B1[1] >= B2[1] else B2

    # experimental angles
    gamma_e = math.atan2(B_mm[1], B_mm[0])
    alpha_e = math.atan2(B_mm[1] - A_mm[1], B_mm[0] - A_mm[0])
    s_e = float(np.dot(A_mm, g))

    # theoretical from no-slip: x_O(t) = R * γ(t)  => γ = (x_O - x_O0)/R
    x_disp = xO_mm - disc_x0_mm
    gamma_t = x_disp / R_MM
    B_th = np.array([R_MM * math.cos(gamma_t), R_MM * math.sin(gamma_t)], float)
    g_dot_B = float(np.dot(g, B_th))
    # solve s from |s g - B_th| = L
    # s^2 - 2(g·B)s + (R^2 - L^2) = 0
    c0 = R_MM**2 - L_MM**2
    bq = -2 * g_dot_B
    discq = bq**2 - 4 * c0
    if discq < 0:
        s_t = g_dot_B  # fallback
    else:
        s1 = (-bq + math.sqrt(discq)) / 2
        s2 = (-bq - math.sqrt(discq)) / 2
        s_t = s1 if abs(s1 - s_e) < abs(s2 - s_e) else s2
    A_th = s_t * g
    alpha_t = math.atan2((B_th - A_th)[1], (B_th - A_th)[0])

    times.append(i * dt)
    gamma_exp.append(gamma_e)
    gamma_th.append(gamma_t)
    s_exp.append(s_e)
    s_th.append(s_t)
    alpha_exp.append(alpha_e)
    alpha_th.append(alpha_t)
    xO_mm_list.append(xO_mm)
    A_mm_list.append(A_mm)

# =========================
# 4. DERIVATIVES & ICZV
# =========================
times = np.array(times)
gamma_exp = np.array(gamma_exp)
gamma_th = np.array(gamma_th)
s_exp = np.array(s_exp)
s_th = np.array(s_th)
alpha_exp = np.array(alpha_exp)
alpha_th = np.array(alpha_th)
xO_mm_list = np.array(xO_mm_list)
A_mm_arr = np.vstack(A_mm_list)

gamma_dot = np.gradient(gamma_exp, dt)
xO_dot = np.gradient(xO_mm_list, dt)
Rgamma_dot = R_MM * gamma_dot

# ICZV
vA = np.gradient(A_mm_arr, dt, axis=0)
alpha_dot = np.gradient(alpha_exp, dt)
ICZV = []
for k in range(len(times)):
    if abs(alpha_dot[k]) < 1e-4:
        ICZV.append(ICZV[-1] if ICZV else np.array([np.nan, np.nan]))
    else:
        vAx, vAy = vA[k]
        rAP = np.array([-vAy, vAx], float) / (alpha_dot[k] ** 2)
        P = A_mm_arr[k] + rAP
        ICZV.append(P)
ICZV = np.vstack(ICZV)

# =========================
# 5. PLOTS
# =========================
out_dir = Path(".")
# 1. gamma
plt.figure()
plt.plot(times, gamma_exp, label="experiment")
plt.plot(times, gamma_th, "--", label="theory")
plt.xlabel("time (s)"); plt.ylabel("γ (rad)")
plt.title("Disc rotation γ(t)")
plt.legend(); plt.tight_layout()
plt.savefig(out_dir / "fig1_gamma.png")
plt.close()

# 2. s(t)
plt.figure()
plt.plot(times, s_exp, label="experiment")
plt.plot(times, s_th, "--", label="theory")
plt.xlabel("time (s)"); plt.ylabel("s (mm)")
plt.title("Slider displacement s(t)")
plt.legend(); plt.tight_layout()
plt.savefig(out_dir / "fig2_slider.png")
plt.close()

# 3. alpha
plt.figure()
plt.plot(times, alpha_exp, label="experiment")
plt.plot(times, alpha_th, "--", label="theory")
plt.xlabel("time (s)"); plt.ylabel("α (rad)")
plt.title("Rod orientation α(t)")
plt.legend(); plt.tight_layout()
plt.savefig(out_dir / "fig3_alpha.png")
plt.close()

# 4. no-slip
plt.figure()
plt.plot(times, xO_dot, label="ẋ_O")
plt.plot(times, Rgamma_dot, "--", label="R γ̇")
plt.xlabel("time (s)"); plt.ylabel("velocity (mm/s)")
plt.title("Verification of no-slip")
plt.legend(); plt.tight_layout()
plt.savefig(out_dir / "fig4_noslip.png")
plt.close()

# 5. ICZV
plt.figure()
plt.plot(ICZV[:,0], ICZV[:,1])
plt.scatter(A_mm_arr[:,0], A_mm_arr[:,1], s=5)
plt.axis("equal")
plt.xlabel("x (mm)"); plt.ylabel("y (mm)")
plt.title("ICZV trajectory")
plt.tight_layout()
plt.savefig(out_dir / "fig5_iczv.png")
plt.close()

print("done, saved 5 figures")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/shavarshmelikyan/opt/anaconda3/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/shavarshmelikyan/opt/anaconda3/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/shavarshmelikyan/opt/anaconda3/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/shavarshmelikyan/opt/anaconda3/lib/python3.9/site-packages/traitlets/config/application.py",

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [2]:
!pip install opencv-python matplotlib numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 MB 4.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 6.2 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.22.4
    Uninstalling numpy-1.22.4:
      Successfully uninstalled numpy-1.22.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [opencv-python]0m [opencv-python]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
daal4py 2021.3.0 requires daal==2021.2.3, which is not installed.
numba 0.54.1 requires numpy<1.21,>=1.17, but you have numpy 2.0.2 which is incompatible.
scipy 1.7.3 requires numpy<1.23.0,>=1.16.5, but you have numpy 2.0.2 which is incompatible.


In [4]:
print('meow')

meow
